<div dir="rtl" style="text-align: right;">
<h2>تمرین ۴: بهبود الگوریتم Canny</h2>
<p>در این بخش نسخه بهبودیافته‌ای از الگوریتم Canny را پیاده‌سازی می‌کنیم که دارای ویژگی‌های زیر است:</p>
<ul>
    <li><b>پیش‌پردازش CLAHE:</b> برای افزایش کنتراست محلی و برجسته کردن <strong>لبه‌های نازک</strong>.</li>
    <li><b>فیلتر Bilateral:</b> برای حذف نویز بدون محو کردن لبه‌ها (بسیار مناسب برای <strong>تصاویر پزشکی</strong> مانند X-Ray یا MRI).</li>
    <li><b>آستانه‌گذاری منطقه‌ای (Regional Adaptive Thresholding):</b> تصویر به بلوک‌های محلی تقسیم شده و مقادیر $T_{min}$ و $T_{max}$ بر اساس میانه روشنایی هر منطقه تنظیم می‌شوند تا لبه‌ها در بخش‌های تاریک و روشن تصویر با دقت یکسانی تشخیص داده شوند.</li>
</ul>
</div>


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def regional_canny(image, block_size=64, sigma=0.33):
    """
    اعمال الگوریتم Canny به صورت منطقه‌ای با تنظیم آستانه‌ها بر اساس میانه محلی
    """
    edges = np.zeros_like(image)
    h, w = image.shape
    
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            # استخراج بلوک محلی
            block = image[i:min(i+block_size, h), j:min(j+block_size, w)]
            
            # محاسبه میانه روشنایی در بلوک
            v = np.median(block)
            
            # تنظیم آستانه‌ها به صورت داینامیک برای همین منطقه
            lower = int(max(0, (1.0 - sigma) * v))
            upper = int(min(255, (1.0 + sigma) * v))
            
            # جلوگیری از نزدیک شدن بیش از حد دو آستانه
            if upper - lower < 20:
                upper = lower + 20
                
            # اعمال Canny روی بلوک و جایگذاری در تصویر نهایی
            block_edges = cv2.Canny(block, lower, upper)
            edges[i:min(i+block_size, h), j:min(j+block_size, w)] = block_edges
            
    return edges

# 1. خواندن تصویر
img = cv2.imread('sample1.jpg')
if img is None:
    print("خطا: تصویر sample1.jpg پیدا نشد! لطفا مسیر عکس را بررسی کنید.")
else:
    # تبدیل به خاکستری
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. پیش پردازش اول: CLAHE برای لبه های نازک و کم کنتراست
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    # 3. پیش پردازش دوم: فیلتر Bilateral برای تصاویر پزشکی (حفظ لبه، حذف نویز)
    blurred = cv2.bilateralFilter(enhanced, d=9, sigmaColor=75, sigmaSpace=75)

    # 4. مقایسه: اعمال Canny استاندارد
    standard_canny = cv2.Canny(gray, 100, 200)

    # 5. Canny بهبود یافته و منطقه‌ای
    improved_canny = regional_canny(blurred, block_size=64, sigma=0.33)

    # 6. رسم نتایج
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(standard_canny, cmap='gray')
    plt.title('Standard Canny (Global)')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(improved_canny, cmap='gray')
    plt.title('Improved Regional Canny')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
